In [17]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [18]:
train = pd.read_csv("../data/processed/train.csv")
validation = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")
print("Train:", train.shape)
display(train.head())
print("Validation:", validation.shape)
display(validation.head())
print("Test:", test.shape)
display(test.head())

Train: (73857, 10)


,reviewers,course_id,rating,date_reviews,title,description,level,skills,user_id,course_idx
0,By H A H,java-programming,4,2020-05-12,Java Programming: Solving Problems with Software,Learn to code in Java and improve your program...,Beginner,"[""Java"", ""Object Oriented Programming (OOP)"", ...",0,295
1,By H A H,machine-learning-with-python,3,2020-05-19,machine-learning-with-python,NaN,Unknown,"[""Feature Engineering"", ""Applied Machine Learn...",0,313
2,By H A H,python-data,5,2020-05-20,Python Data Structures,This course will introduce the core data struc...,Beginner,"[""Data Structures"", ""Development Environment"",...",0,413
3,By H A H,machine-learning,5,2020-08-10,Supervised Machine Learning: Regression and Cl...,In the first course of the Machine Learning Sp...,Beginner,"[""Jupyter"", ""Predictive Modeling"", ""Python Pro...",0,309
4,By H A H,neural-networks-deep-learning,5,2020-08-18,Neural Networks and Deep Learning,In the first course of the Deep Learning Speci...,Intermediate,"[""Calculus"", ""Machine Learning"", ""Artificial N...",0,352


Validation: (6502, 10)


,reviewers,course_id,rating,date_reviews,title,description,level,skills,user_id,course_idx
0,By H A H,nlp-sequence-models,5,2020-09-21,Sequence Models,In the fifth course of the Deep Learning Speci...,Intermediate,"[""Tensorflow"", ""Natural Language Processing"", ...",0,357
1,By N B,what-is-a-proof,5,2018-11-08,Mathematical Thinking in Computer Science,Mathematical thinking is crucial in all areas ...,Beginner,"[""Theoretical Computer Science"", ""Computer Sci...",1,544
2,By 2516_Anirudh G,cybersecurity-compliance-framework-system-admi...,5,2020-08-01,cybersecurity-compliance-framework-system-admi...,NaN,Unknown,"[""Regulatory Compliance"", ""General Data Protec...",2,105
3,By A P,python,5,2020-04-29,Programming for Everybody (Getting Started wit...,This course aims to teach everyone the basics ...,Beginner,"[""Computer Programming"", ""Software Installatio...",3,410
4,By A S,machine-learning-with-python,4,2020-08-08,machine-learning-with-python,NaN,Unknown,"[""Feature Engineering"", ""Applied Machine Learn...",4,313


Test: (12942, 10)


,reviewers,course_id,rating,date_reviews,title,description,level,skills,user_id,course_idx
0,By H A H,ai-for-everyone,5,2020-09-23,AI For Everyone,AI is not only for engineers. If you want your...,Beginner,"[""Strategic Thinking"", ""Artificial Intelligenc...",0,8
1,By H A H,convolutional-neural-networks-tensorflow,4,2020-10-05,Convolutional Neural Networks in TensorFlow,If you are a software developer who wants to b...,Intermediate,"[""Supervised Learning"", ""Deep Learning"", ""Imag...",0,88
2,By N B,wharton-introduction-spreadsheets-models,1,2019-04-21,Introduction to Spreadsheets and Models,The simple spreadsheet is one of the most powe...,Mixed,"[""Regression Analysis"", ""Spreadsheet Software""...",1,536
3,By 2516_Anirudh G,ibm-penetration-testing-incident-response-fore...,5,2020-08-13,"Penetration Testing, Threat Hunting, and Crypt...",The cost of a security breach averages nearly ...,Intermediate,"[""Security Testing"", ""Cybersecurity"", ""Cyber T...",2,243
4,By 2516_Anirudh G,it-security,5,2020-09-25,it-security,NaN,Unknown,"[""Security Awareness"", ""Authorization (Computi...",2,294


In [19]:
print("Các cột train:", train.columns.tolist())
print("Các cột validation:", validation.columns.to_list())
print("Các cột test:", test.columns.tolist())

print("\nThông tin tập train:")
train.info()

Các cột train: ['reviewers', 'course_id', 'rating', 'date_reviews', 'title', 'description', 'level', 'skills', 'user_id', 'course_idx']
Các cột validation: ['reviewers', 'course_id', 'rating', 'date_reviews', 'title', 'description', 'level', 'skills', 'user_id', 'course_idx']
Các cột test: ['reviewers', 'course_id', 'rating', 'date_reviews', 'title', 'description', 'level', 'skills', 'user_id', 'course_idx']

Thông tin tập train:
<class 'pandas.DataFrame'>
RangeIndex: 73857 entries, 0 to 73856
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   reviewers     73857 non-null  str  
 1   course_id     73857 non-null  str  
 2   rating        73857 non-null  int64
 3   date_reviews  73857 non-null  str  
 4   title         73857 non-null  str  
 5   description   62047 non-null  str  
 6   level         73857 non-null  str  
 7   skills        70665 non-null  str  
 8   user_id       73857 non-null  int64
 9   course_idx  

### Baseline 1 — Popularity-Based Recommendation
Đề xuất các khoá học phổ biến nhất dựa trên số lượt đánh giá tích cực trong train

In [20]:

# Ngưỡng xác định đánh giá tích cực
nguong_tich_cuc = 4

# Lọc các lượt đánh giá tích cực trong tập train
positive_train = train[
    train["rating"] >= nguong_tich_cuc
].copy()

# Đếm số lượt đánh giá tích cực của từng khóa học
so_danh_gia_tich_cuc = (
    positive_train.groupby("course_idx")
    .size()
    .rename("so_danh_gia_tich_cuc")
)

# Danh sách tất cả khóa học xuất hiện trong train
tat_ca_khoa_hoc = train["course_idx"].unique()

# Tạo bảng độ phổ biến cho tất cả khóa học
mon_pho_bien = pd.DataFrame({
    "course_idx": tat_ca_khoa_hoc
})

mon_pho_bien["so_danh_gia_tich_cuc"] = (
    mon_pho_bien["course_idx"]
    .map(so_danh_gia_tich_cuc)
    .fillna(0)
    .astype(int)
)

# Xếp hạng theo số lượt đánh giá tích cực giảm dần
mon_pho_bien = (
    mon_pho_bien
    .sort_values(
        "so_danh_gia_tich_cuc",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Số lượt đánh giá tích cực:", len(positive_train))
print("Số khóa học trong train:", train["course_idx"].nunique())
print(
    "Số khóa học có ít nhất 1 đánh giá tích cực:",
    (mon_pho_bien["so_danh_gia_tich_cuc"] > 0).sum()
)

display(mon_pho_bien.head(10))


Số lượt đánh giá tích cực: 68138
Số khóa học trong train: 553
Số khóa học có ít nhất 1 đánh giá tích cực: 553


,course_idx,so_danh_gia_tich_cuc
0,309,1743
1,352,1732
2,410,1633
3,305,1591
4,413,1475
5,118,1114
6,487,1095
7,421,1070
8,546,1044
9,201,905


In [21]:
# Các khóa học mà mỗi người dùng đã tương tác trong train
khoa_hoc_da_hoc = (
    train.groupby("user_id")["course_idx"]
    .apply(set)
    .to_dict()
)

# Danh sách khóa học sắp xếp theo độ phổ biến
danh_sach_pho_bien = mon_pho_bien["course_idx"].tolist()

def goi_y_top_n(user_id, n=10):
    da_tuong_tac = khoa_hoc_da_hoc.get(user_id, set())

    goi_y = [
        course_idx
        for course_idx in danh_sach_pho_bien
        if course_idx not in da_tuong_tac
    ]

    return goi_y[:n]

# Thử đề xuất cho một người dùng trong train
ds_user = train["user_id"].unique()
user_thu = ds_user[5]

print("User ID:", user_thu)
print("Các khóa học đã tương tác:", len(
    khoa_hoc_da_hoc.get(user_thu, set())
))
print("Top-10 đề xuất:", goi_y_top_n(user_thu, 10))

User ID: 5
Các khóa học đã tương tác: 15
Top-10 đề xuất: [309, 352, 410, 305, 413, 118, 487, 421, 546, 201]


In [22]:

# Lọc các tương tác tích cực trong validation
validation_tich_cuc = validation[
    validation["rating"] >= nguong_tich_cuc
].copy()

# Tập khóa học liên quan của từng người dùng trong validation
tap_lien_quan_validation = (
    validation_tich_cuc
    .groupby("user_id")["course_idx"]
    .apply(set)
    .to_dict()
)

# Thống kê người dùng trong validation
so_user_validation = validation["user_id"].nunique()

so_user_co_lien_quan_validation = len(
    tap_lien_quan_validation
)

so_user_khong_co_lien_quan_validation = (
    so_user_validation - so_user_co_lien_quan_validation
)

print("Số user trong validation:", so_user_validation)

print(
    "Số user có ít nhất 1 khóa học tích cực:",
    so_user_co_lien_quan_validation
)

print(
    "Số user không có khóa học tích cực:",
    so_user_khong_co_lien_quan_validation
)


Số user trong validation: 6502
Số user có ít nhất 1 khóa học tích cực: 5986
Số user không có khóa học tích cực: 516


In [23]:
# Tập khóa học liên quan trong test của mỗi người dùng
test_tich_cuc = test[test["rating"] >= nguong_tich_cuc]

tap_lien_quan = (
    test_tich_cuc.groupby("user_id")["course_idx"]
    .apply(set)
    .to_dict()
)

so_user_test = test["user_id"].nunique()
so_user_co_lien_quan = len(tap_lien_quan)
so_user_khong_co_lien_quan = so_user_test - so_user_co_lien_quan

print("Số user trong test:", so_user_test)
print("Số user có ít nhất 1 khóa học tích cực:", so_user_co_lien_quan)
print("Số user không có khóa học tích cực:", so_user_khong_co_lien_quan)

Số user trong test: 6540
Số user có ít nhất 1 khóa học tích cực: 6443
Số user không có khóa học tích cực: 97


In [24]:

def precision_at_k(goi_y, lien_quan, k=10):
    top_k = goi_y[:k]

    if len(top_k) == 0:
        return 0.0

    so_khoa_hoc_dung = len(set(top_k) & lien_quan)

    return so_khoa_hoc_dung / k


def recall_at_k(goi_y, lien_quan, k=10):
    if len(lien_quan) == 0:
        return np.nan

    top_k = goi_y[:k]
    so_khoa_hoc_dung = len(set(top_k) & lien_quan)

    return so_khoa_hoc_dung / len(lien_quan)


def f1_at_k(goi_y, lien_quan, k=10):
    precision = precision_at_k(goi_y, lien_quan, k)
    recall = recall_at_k(goi_y, lien_quan, k)

    if np.isnan(recall) or precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def ndcg_at_k(goi_y, lien_quan, k=10):
    if len(lien_quan) == 0:
        return np.nan

    top_k = goi_y[:k]

    dcg = sum(
        1 / math.log2(i + 2)
        for i, course_idx in enumerate(top_k)
        if course_idx in lien_quan
    )

    so_relevant_toi_da = min(len(lien_quan), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(so_relevant_toi_da)
    )

    return dcg / idcg if idcg > 0 else np.nan


In [25]:
# Đánh giá Popularity-Based Recommendation trên validation

K = 10

ket_qua_validation = []

for user_id, lien_quan in tap_lien_quan_validation.items():
    # Sinh Top-K khóa học cho user
    goi_y = goi_y_top_n(user_id, n=K)

    # Tính các metric
    precision = precision_at_k(goi_y, lien_quan, k=K)
    recall = recall_at_k(goi_y, lien_quan, k=K)
    f1 = f1_at_k(goi_y, lien_quan, k=K)
    ndcg = ndcg_at_k(goi_y, lien_quan, k=K)

    ket_qua_validation.append({
        "user_id": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "F1@10": f1,
        "NDCG@10": ndcg
    })

df_ket_qua_validation = pd.DataFrame(ket_qua_validation)

print("Số user được đánh giá:", len(df_ket_qua_validation))
print("\nKết quả trung bình trên validation:")

display(
    df_ket_qua_validation[
        ["Precision@10", "Recall@10", "F1@10", "NDCG@10"]
    ].mean().to_frame("Giá trị trung bình")
)

Số user được đánh giá: 5986

Kết quả trung bình trên validation:


,Giá trị trung bình
Precision@10,0.020197
Recall@10,0.201971
F1@10,0.036722
NDCG@10,0.100343


In [26]:
# Đánh giá Popularity-Based Recommendation trên test

K = 10
ket_qua_test = []

for user_id, lien_quan in tap_lien_quan.items():
    goi_y = goi_y_top_n(user_id, n=K)

    ket_qua_test.append({
        "user_id": user_id,
        "Precision@10": precision_at_k(goi_y, lien_quan, k=K),
        "Recall@10": recall_at_k(goi_y, lien_quan, k=K),
        "F1@10": f1_at_k(goi_y, lien_quan, k=K),
        "NDCG@10": ndcg_at_k(goi_y, lien_quan, k=K)
    })

df_ket_qua_test = pd.DataFrame(ket_qua_test)

print("Số user được đánh giá:", len(df_ket_qua_test))
print("\nKết quả trung bình trên test:")

display(
    df_ket_qua_test[
        ["Precision@10", "Recall@10", "F1@10", "NDCG@10"]
    ].mean().to_frame("Giá trị trung bình")
)

Số user được đánh giá: 6443

Kết quả trung bình trên test:


,Giá trị trung bình
Precision@10,0.034611
Recall@10,0.188344
F1@10,0.058149
NDCG@10,0.114531
